# Conformal Model Evaluation and Diagnostics

This notebook evaluates the performance of the trained fake review detector. It reads the serialized conformal classifier, analyzes the test data split, computes classification metrics (ROC, AUC, Precision-Recall, Confusion Matrix), examines the conformal prediction sets, and visualizes how model probabilities correlate with review features.

## 1. How to Train the Model

To retrain the model and log metrics to MLflow, run the training script from the repository root:
```bash
uv run python -m src.train
```

### Launching the MLflow UI
To view your experiments, parameters, and logged metrics in MLflow, run this command in your terminal:
```bash
uv run mlflow ui --port 5000
```
If you are in GitHub Codespaces, a popup will appear allowing you to open the port in your browser.

In [ ]:
import os
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_curve, 
    auc, 
    precision_recall_curve, 
    average_precision_score
)

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

## 2. Load Model and Splits

We load the pickled `ConformalClassifier` model and the processed `test.parquet` split.

In [ ]:
model_path = Path("../models/conformal_model.pkl")
test_path = Path("../data/processed/test.parquet")

# Fallback if running from repo root
if not model_path.exists():
    model_path = Path("models/conformal_model.pkl")
    test_path = Path("data/processed/test.parquet")

if not model_path.exists() or not test_path.exists():
    raise FileNotFoundError("Model or test data splits not found! Please run 'uv run python -m src.train' first.")

with open(model_path, "rb") as f:
    conformal_model = pickle.load(f)

df_test = pd.read_parquet(test_path)
print("Model loaded successfully!")
print(f"Test set size: {len(df_test):,} rows")

## 3. Standard Classification Metrics

Let's extract predictions from our base pipeline and analyze standard classification performance metrics.

In [ ]:
X_test = df_test.drop(columns=["label"])
y_test = df_test["label"]

# Predict probabilities from the underlying scikit-learn estimator
# probs[:, 1] represents the probability of being Fake (1)
probs = conformal_model.estimator.predict_proba(X_test)
y_scores = probs[:, 1]
y_pred = conformal_model.estimator.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Genuine", "Fake"]))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Genuine", "Fake"], yticklabels=["Genuine", "Fake"])
plt.title("Confusion Matrix")
plt.ylabel("Actual Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()

### 3.1 ROC Curve and Precision-Recall Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_scores)
roc_auc = auc(fpr, tpr)

precision, recall, _ = precision_recall_curve(y_test, y_scores)
avg_precision = average_precision_score(y_test, y_scores)

plt.figure(figsize=(14, 6))

# ROC Curve
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC)")
plt.legend(loc="lower right")

# Precision-Recall Curve
plt.subplot(1, 2, 2)
plt.plot(recall, precision, color="blue", lw=2, label=f"PR curve (AP = {avg_precision:.4f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend(loc="lower left")

plt.tight_layout()
plt.show()

## 4. Conformal Prediction Set Analysis

Conformal prediction outputs set predictions rather than simple labels. Let's inspect the coverage and size distributions.

In [ ]:
pred_sets, _ = conformal_model.predict_set(X_test)
y_test_arr = y_test.values

# Empirical Coverage
in_set = [y_test_arr[i] in pred_sets[i] for i in range(len(y_test_arr))]
coverage = np.mean(in_set)
print(f"Target Confidence:  {100 * (1 - conformal_model.alpha):.1f}%")
print(f"Empirical Coverage: {coverage:.2%}")

# Set size distribution
set_sizes = [len(s) for s in pred_sets]
df_sets = pd.DataFrame({
    "set_size": set_sizes,
    "pred_set_str": [str(s) for s in pred_sets]
})

set_counts = df_sets["pred_set_str"].value_counts()
plt.figure(figsize=(7, 5))
sns.barplot(x=set_counts.index, y=set_counts.values, hue=set_counts.index, palette="viridis", legend=False)
plt.title("Conformal Prediction Set Frequency")
plt.xlabel("Prediction Set")
plt.ylabel("Count")
for i, val in enumerate(set_counts.values):
    plt.text(i, val + (max(set_counts.values)*0.01), f"{val:,}\n({val/len(df_test)*100:.1f}%)", ha="center", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

### 4.1 Examples of Uncertain Reviews
Prediction sets of `{0, 1}` signify that the model is **uncertain** (unable to reject either hypothesis at the 95% confidence level).

In [ ]:
uncertain_mask = [len(s) == 2 for s in pred_sets]
df_uncertain = df_test[uncertain_mask]

print(f"Found {len(df_uncertain):,} uncertain reviews ({len(df_uncertain)/len(df_test):.2%} of test set).\n")
if len(df_uncertain) > 0:
    sample_uncertain = df_uncertain.head(3)
    for i, (idx, row) in enumerate(sample_uncertain.iterrows()):
        print(f"Uncertain Example #{i+1}:")
        print(f"  Actual Label: {'Fake' if row['label'] == 1 else 'Genuine'}")
        print(f"  Text:         {row['clean_text'][:200]}...")
        # Print probabilities
        row_features = pd.DataFrame([row.drop('label')])
        prob_fake = conformal_model.estimator.predict_proba(row_features)[0, 1]
        print(f"  P(Fake):      {prob_fake:.4f}")
        print("-" * 80)

## 5. Visualizing Average Probability by Feature

We investigate how the probability of being fake ($P(\text{Fake})$) correlates with our engineered structural features. We group the continuous variables into bins (quantiles) and compute the average model probability for each bin.

In [ ]:
# Add probabilities to the test dataframe for analysis
df_analysis = df_test.assign(prob_fake=y_scores)

def plot_prob_by_feature_bins(df_data, feature_col, num_bins=5):
    # Bin the feature into equal-frequency quantiles
    # Use qcut to get roughly equal number of points per bin
    try:
        df_data = df_data.assign(
            bin=pd.qcut(df_data[feature_col], q=num_bins, duplicates='drop')
        )
    except Exception as e:
        # Fallback to standard cut if qcut fails
        df_data = df_data.assign(
            bin=pd.cut(df_data[feature_col], bins=num_bins)
        )
    
    # Compute average probability of fake per bin
    bin_stats = df_data.groupby("bin", observed=False).agg(
        avg_prob_fake=("prob_fake", "mean"),
        actual_fake_ratio=("label", "mean"),
        count=("label", "count")
    ).reset_index()

    # Plotting
    plt.figure(figsize=(10, 5))
    x_labels = bin_stats["bin"].astype(str)
    
    # Plot predicted average vs actual ratio
    x = np.arange(len(bin_stats))
    width = 0.35
    
    plt.bar(x - width/2, bin_stats["avg_prob_fake"], width, label="Average Model P(Fake)", color="salmon")
    plt.bar(x + width/2, bin_stats["actual_fake_ratio"], width, label="Actual Fake Ratio", color="skyblue")
    
    plt.xticks(x, x_labels, rotation=15)
    plt.xlabel(f"{feature_col} Bins")
    plt.ylabel("Probability / Ratio")
    plt.title(f"Model Prediction Accuracy vs {feature_col} (Binned)")
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    return bin_stats

### 5.1 Average Probability by Review Length (Word Count)

In [ ]:
word_count_stats = plot_prob_by_feature_bins(df_analysis, "word_count", num_bins=5)
word_count_stats

### 5.2 Average Probability by Capitalization Ratio

In [ ]:
cap_ratio_stats = plot_prob_by_feature_bins(df_analysis, "cap_ratio", num_bins=5)
cap_ratio_stats

### 5.3 Average Probability by Exclamation Mark Ratio

In [ ]:
excl_ratio_stats = plot_prob_by_feature_bins(df_analysis, "exclamation_ratio", num_bins=5)
excl_ratio_stats

## 6. Summary and Modeling Strategy Next Steps

* **Conformal Threshold Validation**: Our target significance is $\alpha = 0.05$ (meaning we expect at least 95.0% coverage). Compare the empirical coverage above: if it is $\approx 95\%$, then the conformal calibration holds true.
* **Uncertainty Sets (Size 2)**: Sets of `{0, 1}` represent cases where the classifier's predicted probability is close to $0.5$ or falls inside the calibration region of uncertainty. These reviews represent cases we would flag for human moderation rather than automatically penalizing/approving.
* **Model Refinements**: If the AUC or Precision/Recall is insufficient, we can tune hyperparameters ($C$ regularization parameter, $N$-gram range, TF-IDF max features) or experiment with other classifiers in `src/train.py`, and log them to MLflow to track performance improvements.